# Laboratorio — Clases desbalanceadas (15 min)
**Curso:** Aprendizaje Automático · MCDA 2026-1
**Sesión:** Clases desbalanceadas: métricas, remuestreo y umbral

## Objetivo
Detectar el problema del desbalance, ver por qué la *accuracy* engaña, y aplicar tres estrategias clásicas para mejorar la detección de la clase minoritaria.

## Dataset
Generamos un dataset sintético de **detección de fraude** con `make_classification`: 10 000 transacciones, 20 variables y solo **2 % de fraudes** (clase 1).

## Agenda (15 min)
| Paso | Tiempo | Qué haces |
|------|--------|-----------|
| 1. Crear datos y ver el desbalance | 2 min | Conteo y gráfico |
| 2. Baseline ingenuo | 3 min | Logística sin tocar nada |
| 3. Métricas adecuadas | 2 min | Precision, recall, F1, PR-AUC |
| 4. Estrategias contra el desbalance | 4 min | `class_weight`, oversampling, umbral |
| 5. Retos | 4 min | 3 micro-retos individuales |

## Paso 1 · Crear datos y ver el desbalance (2 min)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=10_000,
    n_features=20,
    n_informative=6,
    n_redundant=4,
    weights=[0.98, 0.02],   # 2 % de la clase 1 (fraude)
    flip_y=0.01,
    class_sep=1.2,
    random_state=42,
)

print('Forma X:', X.shape)
print('Distribución de clases:', dict(zip(*np.unique(y, return_counts=True))))
print(f'% clase positiva (fraude): {y.mean()*100:.2f} %')

pd.Series(y).value_counts().plot.bar(color=['#1FA8DC', '#E84B4B'])
plt.title('Distribución de clases'); plt.xlabel('Clase'); plt.ylabel('Conteo')
plt.xticks(rotation=0); plt.show()

**Pregunta rápida:** si un modelo dijera *"todo es 0"*, ¿qué accuracy obtendría? ¿Sirve ese modelo para detectar fraude?

## Paso 2 · Baseline ingenuo (3 min)
Entrenamos una regresión logística *sin* tocar el desbalance y miramos la accuracy.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

base = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=5000, random_state=42))
])
base.fit(X_train, y_train)

y_pred_base = base.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred_base):.3f}')
print('Matriz de confusión [filas=real, cols=pred]:')
print(confusion_matrix(y_test, y_pred_base))

**Observa:** la accuracy se ve altísima, pero la columna *predijo 1* casi no tiene casos. El modelo aprendió que decir "no fraude" siempre es lo más rentable.

## Paso 3 · Métricas adecuadas para desbalance (2 min)
La accuracy miente cuando hay desbalance. Usamos *precision*, *recall*, *F1* y **PR-AUC** (área bajo la curva precisión-recall), que se enfocan en la clase positiva.

In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             average_precision_score, classification_report,
                             PrecisionRecallDisplay, ConfusionMatrixDisplay)

def evaluar(modelo, nombre, threshold=0.5):
    proba = modelo.predict_proba(X_test)[:, 1]
    pred  = (proba >= threshold).astype(int)
    print(f'--- {nombre} (umbral={threshold}) ---')
    print(f'Precision: {precision_score(y_test, pred, zero_division=0):.3f}')
    print(f'Recall   : {recall_score(y_test, pred):.3f}')
    print(f'F1       : {f1_score(y_test, pred):.3f}')
    print(f'PR-AUC   : {average_precision_score(y_test, proba):.3f}')
    print()
    return proba, pred

proba_base, pred_base = evaluar(base, 'Baseline (sin balanceo)')

## Paso 4 · Estrategias contra el desbalance (4 min)

Probamos tres recetas, de la más barata a la más costosa:

1. **`class_weight='balanced'`** — penaliza más los errores en la clase minoritaria.
2. **Oversampling con SMOTE** — generamos ejemplos sintéticos de la clase minoritaria.
3. **Ajustar el umbral de decisión** — bajamos el corte de 0.5 a un valor que mejore el recall.

In [ ]:
# (1) class_weight balanced
weighted = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=5000, class_weight='balanced',
                                  random_state=42))
])
weighted.fit(X_train, y_train)
proba_w, pred_w = evaluar(weighted, 'class_weight=balanced')

In [ ]:
# (2) Oversampling con SMOTE
# Si imbalanced-learn no está instalado, la celda lo instala
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'imbalanced-learn', '-q'])
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline

smote_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42)),
    ('clf',    LogisticRegression(max_iter=5000, random_state=42))
])
smote_pipe.fit(X_train, y_train)
proba_s, pred_s = evaluar(smote_pipe, 'SMOTE + Logística')

In [ ]:
# (3) Ajustar el umbral del baseline para maximizar F1
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y_test, proba_base)
f1s = 2 * prec * rec / np.where(prec + rec == 0, 1, prec + rec)
best_idx = int(np.argmax(f1s[:-1]))   # el último elemento no tiene umbral asociado
best_thr = thr[best_idx]
print(f'Mejor umbral según F1: {best_thr:.3f}  ·  F1={f1s[best_idx]:.3f}')

evaluar(base, 'Baseline + umbral ajustado', threshold=best_thr)

# Curva PR comparativa
fig, ax = plt.subplots(figsize=(6, 4))
PrecisionRecallDisplay.from_predictions(y_test, proba_base, ax=ax, name='Baseline')
PrecisionRecallDisplay.from_predictions(y_test, proba_w,    ax=ax, name='class_weight')
PrecisionRecallDisplay.from_predictions(y_test, proba_s,    ax=ax, name='SMOTE')
ax.set_title('Curvas Precisión-Recall'); plt.tight_layout(); plt.show()

## Paso 5 · Retos individuales (4 min)

### Reto 1 · Random Forest balanceado
Reemplaza la regresión logística por `RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)` y evalúalo. ¿Mejora el F1 y el PR-AUC frente al baseline?

### Reto 2 · Sub-muestrear la mayoritaria
Importa `RandomUnderSampler` de `imblearn.under_sampling` y crea un pipeline con *undersampling* en lugar de SMOTE. Compara el recall y la cantidad de filas usadas en entrenamiento.

### Reto 3 · ¿Qué métrica reportarías al negocio?
Si el costo de no detectar un fraude es 100× el costo de revisar un caso legítimo, ¿prefieres maximizar precision, recall o F1? Justifica con una línea y propón un umbral que refleje esa decisión.

In [ ]:
# Reto 1 — tu código aquí


In [ ]:
# Reto 2 — tu código aquí
# pista: from imblearn.under_sampling import RandomUnderSampler


In [ ]:
# Reto 3 — tu respuesta y código aquí


## Cierre
La accuracy puede ser engañosa cuando una clase domina. En problemas desbalanceados conviene mirar **precision, recall, F1 y PR-AUC**, y combinar al menos una estrategia de remuestreo o ponderación con un **ajuste de umbral** alineado con el costo del error en cada clase.